# 대량성 지표

짧은 기간에 지원이 몰리는 행동을 지원량으로 확인한다. 7일, 14일, 21일, 30일의 최대 지원 횟수를 비교하고, 사분위 범위와 상위 5% 수준, 연도별 결과를 확인해 기간과 횟수 기준을 정한다.

최종 기준은 **유저별 연속 7일 내 최대 지원 횟수가 14건 이상이면 1점, 미만이면 0점**이다.


## 1. 분석 데이터

지원 시각은 원본의 UTC 기준에서 한국시간으로 변환하고, 한국시간 기준 2022년과 2023년 지원만 남긴다. 지원량을 계산하는 분석이므로 지원이 많다는 이유로 유저를 먼저 제거하지 않는다.

- application_uuid: 지원 기록의 ID
- user_uuid: 유저 ID. 다른 담당자의 점수표와 연결할 때도 사용한다.
- cdate_kst: 한국시간으로 변환한 지원 시각

같은 유저의 지원 시각이 순서대로 놓이도록 정렬한다. 뒤에서 유저별로 직전 기간의 지원 횟수를 계산할 때 이 순서가 필요하다.

In [1]:
import pandas as pd

applications = pd.read_csv(
    './data/Application.csv',
    usecols=['application_uuid', 'user_uuid', 'cdate'],
    dtype={'application_uuid': str, 'user_uuid': str}
)

applications['cdate'] = pd.to_datetime(applications['cdate'], utc=True)
applications['cdate_kst'] = applications['cdate'].dt.tz_convert('Asia/Seoul')

applications = applications[
    applications['cdate_kst'].dt.year.isin([2022, 2023])
].copy()

applications = applications.sort_values(
    ['user_uuid', 'cdate_kst']
).reset_index(drop=True)
applications['year'] = applications['cdate_kst'].dt.year

print('분석 지원 건수:', len(applications))
print('분석 유저 수:', applications['user_uuid'].nunique())
print('지원 ID 중복 건수:', applications['application_uuid'].duplicated().sum())


분석 지원 건수: 92526
분석 유저 수: 10287
지원 ID 중복 건수: 0


## 2. 기간별 최대 지원 횟수

2년 누적 지원 횟수만 보면 오랫동안 꾸준히 지원한 유저와 짧게 몰아서 지원한 유저가 섞인다. 이를 구분하기 위해 각 지원 시각에서 직전 7일, 14일, 21일, 30일의 지원 횟수를 계산한다.

**applications_in_7d**는 한 지원 시각에서 계산한 직전 7일 지원 횟수다. **max_7d**는 그 유저의 모든 지원 시각 중 이 값이 가장 컸던 횟수다.

- application_count에 1을 넣어 각 행을 지원 1건으로 센다.
- groupby는 유저별로 데이터를 나눈다.
- rolling은 각 지원 시각을 기준으로 지정한 기간을 움직이며 계산한다.
- sum은 그 기간 안에 있는 지원 기록의 1을 모두 더한다.
- max는 유저별로 가장 지원이 많았던 기간의 횟수를 남긴다.

7일은 월요일부터 일요일까지의 달력상 한 주가 아니라 **각 지원 시각 기준 직전 168시간**이다. 현재 지원은 포함하고, 정확히 7일 전인 시각은 제외한다.



In [2]:
period_days = [7, 14, 21, 30]
applications['application_count'] = 1

for days in period_days:
    count_column = f'applications_in_{days}d'

    applications[count_column] = (
        applications.groupby('user_uuid')
        .rolling(
            f'{days}D',
            on='cdate_kst',
            closed='right'
        )['application_count']
        .sum()
        .to_numpy()
    )

user_period_counts = applications.groupby('user_uuid').agg(
    max_7d=('applications_in_7d', 'max'),
    max_14d=('applications_in_14d', 'max'),
    max_21d=('applications_in_21d', 'max'),
    max_30d=('applications_in_30d', 'max')
)
user_period_counts = user_period_counts.astype(int)

print(user_period_counts.head().to_string())
print('\n기간별 최대 지원 횟수의 백분위')
print(user_period_counts.quantile([0.5, 0.9, 0.95, 0.975]).to_string())


                                      max_7d  max_14d  max_21d  max_30d
user_uuid                                                              
0002535c-eacb-456b-a620-92c917332ba3       1        1        1        1
000b55ad-5c86-438b-a5b0-9ea3a2917789       3        3        3        3
0017a738-4fcc-441e-983b-a6033890a9db       1        1        1        1
003a83bd-6cb7-4f99-ab3a-66904df762e2       1        1        1        1
00495153-f908-4523-9d0d-5b93a57196a4       2        2        2        2

기간별 최대 지원 횟수의 백분위
       max_7d  max_14d  max_21d  max_30d
0.500    2.00      2.0      2.0      3.0
0.900    9.00     10.0     11.0     12.0
0.950   14.00     16.0     18.0     19.0
0.975   20.85     24.0     26.0     29.0


## 3. 지원 횟수의 기준 비교

기간별 최대 지원 횟수에서 Q1, Q3, IQR을 계산한다. Q1은 25백분위, Q3는 75백분위이며 **IQR = Q3 - Q1**이다.

두 가지 사분위 기준을 비교한다.

- Q3 + 1.5 × IQR: 일반적인 이상치 경계
- Q3 + 3 × IQR: 더 큰 값을 구분하는 극단값 경계

경계를 초과하는 지원 횟수를 표시한다. 지원 횟수는 정수이므로 int(경계) + 1로 첫 정수를 구한다. 경계가 13건이면 최종 조건은 14건 이상이 된다.

상위 5% 수준도 함께 비교한다. 이 표의 **p95_cutoff**는 95백분위를 초과하는 첫 정수다. 95백분위 값과 그 값을 초과하는 기준은 구분해야 한다. 95백분위가 14건이면 이 비교 기준은 15건 이상이다.

횟수가 같은 유저가 여럿 있어 실제 해당 비율은 정확히 5%가 아닐 수 있다. IQR 기준과 상위 5% 수준이 비슷한 범위를 가리키는지 확인하는 용도로 사용한다.


In [3]:
period_comparison_rows = []

for days in period_days:
    maximum_counts = user_period_counts[f'max_{days}d']
    user_count = len(maximum_counts)

    q1 = maximum_counts.quantile(0.25)
    q3 = maximum_counts.quantile(0.75)
    iqr = q3 - q1

    iqr_1_5_boundary = q3 + 1.5 * iqr
    iqr_3_boundary = q3 + 3 * iqr
    iqr_1_5_cutoff = int(iqr_1_5_boundary) + 1
    iqr_3_cutoff = int(iqr_3_boundary) + 1

    p95_value = maximum_counts.quantile(0.95)
    p95_cutoff = int(p95_value) + 1

    iqr_1_5_users = (maximum_counts >= iqr_1_5_cutoff).sum()
    iqr_3_users = (maximum_counts >= iqr_3_cutoff).sum()
    p95_users = (maximum_counts >= p95_cutoff).sum()

    period_comparison_rows.append({
        'period_days': days,
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'iqr_1_5_boundary': iqr_1_5_boundary,
        'iqr_1_5_cutoff': iqr_1_5_cutoff,
        'iqr_1_5_users': iqr_1_5_users,
        'iqr_1_5_user_pct': iqr_1_5_users / user_count * 100,
        'iqr_3_boundary': iqr_3_boundary,
        'iqr_3_cutoff': iqr_3_cutoff,
        'iqr_3_users': iqr_3_users,
        'iqr_3_user_pct': iqr_3_users / user_count * 100,
        'p95_value': p95_value,
        'p95_cutoff': p95_cutoff,
        'p95_users': p95_users,
        'p95_user_pct': p95_users / user_count * 100
    })

period_comparison = pd.DataFrame(period_comparison_rows)

print('기간별 사분위 기준')
print(period_comparison[[
    'period_days', 'q1', 'q3', 'iqr',
    'iqr_1_5_cutoff', 'iqr_1_5_users', 'iqr_1_5_user_pct',
    'iqr_3_cutoff', 'iqr_3_users', 'iqr_3_user_pct'
]].round(2).to_string(index=False))

print('\n상위 5% 수준과 3×IQR 기준 비교')
print(period_comparison[[
    'period_days', 'p95_value', 'p95_cutoff',
    'p95_users', 'p95_user_pct',
    'iqr_3_cutoff', 'iqr_3_users', 'iqr_3_user_pct'
]].round(2).to_string(index=False))


기간별 사분위 기준
 period_days  q1  q3  iqr  iqr_1_5_cutoff  iqr_1_5_users  iqr_1_5_user_pct  iqr_3_cutoff  iqr_3_users  iqr_3_user_pct
           7 1.0 4.0  3.0               9           1107             10.76            14          545            5.30
          14 1.0 5.0  4.0              12            860              8.36            18          452            4.39
          21 1.0 6.0  5.0              14            792              7.70            22          365            3.55
          30 1.0 6.0  5.0              14            892              8.67            22          427            4.15

상위 5% 수준과 3×IQR 기준 비교
 period_days  p95_value  p95_cutoff  p95_users  p95_user_pct  iqr_3_cutoff  iqr_3_users  iqr_3_user_pct
           7       14.0          15        487          4.73            14          545            5.30
          14       16.0          17        504          4.90            18          452            4.39
          21       18.0          19        475          4.62    

위 결과에서 7일의 Q1은 1건, Q3는 4건, IQR은 3건이다. 3×IQR 경계는 13건이므로, **14건 이상**이 된다.

7일 기준 1.5×IQR은 9건 이상, 3×IQR은 14건 이상이다. 이번 지표에서는 일반적인 지원량보다 크게 벗어난 행동에 1점을 주기 위해 더 좁은 범위인 3×IQR 기준을 사용한다.

3×IQR을 사용한다고 실제 허수지원자가 확인되는 것은 아니다. 대량성의 배점 범위를 정하는 분포 기준이다.

## 4. 연도별 비교

전체 데이터에서 만든 기준이 한 해의 분포에만 의존하는지 확인하기 위해 2022년과 2023년을 나누어 계산한다.

각 연도의 지원 기록에서 기간별 최대 지원 횟수를 다시 계산한다. 전체 데이터에서 계산한 max_7d를 연도별로 단순 분류하는 방식이 아니다. 따라서 연도별 비교에서는 연말을 넘어가는 지원 기록이 섞이지 않는다.

**year_period_counts**는 연도를 키로 가진 딕셔너리다. year_period_counts[2022]에는 2022년 유저별 최대 지원 횟수표가 들어간다.


In [4]:
year_period_counts = {}

for year in [2022, 2023]:
    year_applications = applications[
        applications['year'] == year
    ].copy().reset_index(drop=True)

    for days in period_days:
        count_column = f'applications_in_{days}d'

        year_applications[count_column] = (
            year_applications.groupby('user_uuid')
            .rolling(
                f'{days}D',
                on='cdate_kst',
                closed='right'
            )['application_count']
            .sum()
            .to_numpy()
        )

    year_period_counts[year] = year_applications.groupby('user_uuid').agg(
        max_7d=('applications_in_7d', 'max'),
        max_14d=('applications_in_14d', 'max'),
        max_21d=('applications_in_21d', 'max'),
        max_30d=('applications_in_30d', 'max')
    ).astype(int)

    print(year, '년 분석 유저 수:', len(year_period_counts[year]))


2022 년 분석 유저 수: 5846
2023 년 분석 유저 수: 5725


### 같은 IQR 규칙을 연도별로 적용

각 연도의 최대 지원 횟수에서 Q1과 Q3를 새로 구하고, 같은 Q3 + 3×IQR 규칙을 적용한다.

**year_period_comparison**에는 연도별 횟수 기준, 해당 유저 수, 해당 비율이 들어간다. 비율의 분모는 해당 연도에 지원한 유저 수다. 한 사람이 두 해에 모두 지원할 수 있으므로 연도별 유저 수를 더해 전체 유저 수로 해석하지 않는다.


In [5]:
year_comparison_rows = []

for year in [2022, 2023]:
    for days in period_days:
        maximum_counts = year_period_counts[year][f'max_{days}d']
        user_count = len(maximum_counts)

        q1 = maximum_counts.quantile(0.25)
        q3 = maximum_counts.quantile(0.75)
        iqr = q3 - q1
        iqr_3_cutoff = int(q3 + 3 * iqr) + 1

        selected_users = (maximum_counts >= iqr_3_cutoff).sum()

        year_comparison_rows.append({
            'year': year,
            'period_days': days,
            'q1': q1,
            'q3': q3,
            'iqr': iqr,
            'iqr_3_cutoff': iqr_3_cutoff,
            'selected_users': selected_users,
            'selected_user_pct': selected_users / user_count * 100
        })

year_period_comparison = pd.DataFrame(year_comparison_rows)

print(year_period_comparison.round(2).to_string(index=False))


 year  period_days  q1  q3  iqr  iqr_3_cutoff  selected_users  selected_user_pct
 2022            7 1.0 4.0  3.0            14             302               5.17
 2022           14 1.0 5.0  4.0            18             261               4.46
 2022           21 1.0 5.0  4.0            18             303               5.18
 2022           30 1.0 5.0  4.0            18             330               5.64
 2023            7 1.0 4.0  3.0            14             293               5.12
 2023           14 1.0 6.0  5.0            22             167               2.92
 2023           21 1.0 6.0  5.0            22             196               3.42
 2023           30 1.0 7.0  6.0            26             179               3.13


### 전체 데이터의 비교 기준을 연도별로 적용

앞의 표는 연도별로 횟수 기준 자체가 유지되는지 확인한 것이다. 아래에서는 전체 데이터에서 정한 기간별 기준을 고정하고, 두 해에 같은 기준을 적용했을 때 해당 비율을 비교한다.

**reference_cutoff**는 전체 데이터에서 구한 상위 5% 비교 기준 또는 3×IQR 기준이다. 각 연도의 해당 비율은 그 연도의 유저별 최대 지원 횟수로 계산한다.

이를 통해 같은 기준을 적용했을 때의 비율과, 매년 기준을 다시 계산했을 때의 변화까지 구분해 볼 수 있다.


In [6]:
fixed_cutoff_rows = []

for _, period_result in period_comparison.iterrows():
    days = int(period_result['period_days'])

    for method, cutoff_column in [
        ('상위 5% 비교', 'p95_cutoff'),
        ('3×IQR', 'iqr_3_cutoff')
    ]:
        reference_cutoff = int(period_result[cutoff_column])

        for year in [2022, 2023]:
            maximum_counts = year_period_counts[year][f'max_{days}d']
            selected_users = (maximum_counts >= reference_cutoff).sum()

            fixed_cutoff_rows.append({
                'method': method,
                'period_days': days,
                'cutoff': reference_cutoff,
                'year': year,
                'selected_users': selected_users,
                'selected_user_pct': selected_users / len(maximum_counts) * 100
            })

year_fixed_cutoff_comparison = pd.DataFrame(fixed_cutoff_rows)

for method in ['상위 5% 비교', '3×IQR']:
    print(method)
    print(year_fixed_cutoff_comparison[
        year_fixed_cutoff_comparison['method'] == method
    ].round(2).to_string(index=False))
    print()


상위 5% 비교
  method  period_days  cutoff  year  selected_users  selected_user_pct
상위 5% 비교            7      15  2022             269               4.60
상위 5% 비교            7      15  2023             261               4.56
상위 5% 비교           14      17  2022             287               4.91
상위 5% 비교           14      17  2023             262               4.58
상위 5% 비교           21      19  2022             272               4.65
상위 5% 비교           21      19  2023             250               4.37
상위 5% 비교           30      20  2022             277               4.74
상위 5% 비교           30      20  2023             264               4.61

3×IQR
method  period_days  cutoff  year  selected_users  selected_user_pct
 3×IQR            7      14  2022             302               5.17
 3×IQR            7      14  2023             293               5.12
 3×IQR           14      18  2022             261               4.46
 3×IQR           14      18  2023             235               4.10


## 5. 지원 활동의 날짜 간격

같은 유저가 지원한 서로 다른 날짜를 순서대로 놓고, 다음 지원일까지의 간격을 계산한다.

먼저 dt.floor('D')로 한국시간의 시각을 그날 0시로 맞춘다. 유저 ID와 날짜가 같은 행을 drop_duplicates로 한 번만 남겨, 하루에 여러 건 지원해도 활동일은 하루로 센다.

groupby(...).diff()는 같은 유저의 이전 지원일과 현재 지원일의 차이를 구한다. 첫 지원일은 이전 날짜가 없으므로 결측값이 되고, 간격을 계산할 때 제외한다.

중앙값과 기간별 누적 비율은 주 단위로 지원 활동을 볼 만한지 확인하는 보조 근거다. 이 비율은 유저의 비율이 아니라 **서로 다른 지원일 사이 간격의 비율**이다.


In [7]:
application_days = applications[['user_uuid', 'cdate_kst']].copy()
application_days['application_day'] = (
    application_days['cdate_kst'].dt.floor('D')
)
application_days = application_days[
    ['user_uuid', 'application_day']
].drop_duplicates()
application_days = application_days.sort_values(
    ['user_uuid', 'application_day']
)

application_days['gap_days'] = (
    application_days.groupby('user_uuid')['application_day']
    .diff()
    .dt.days
)
application_gaps = application_days['gap_days'].dropna()

print('지원일 사이 간격 중앙값:', application_gaps.median(), '일')
print('지원일 사이 간격 수:', len(application_gaps))

for days in period_days:
    gap_pct = (application_gaps <= days).mean() * 100
    print(days, '일 이내 다음 지원일 비율:', round(gap_pct, 1), '%')


지원일 사이 간격 중앙값: 6.0 일
지원일 사이 간격 수: 29683
7 일 이내 다음 지원일 비율: 63.1 %
14 일 이내 다음 지원일 비율: 78.5 %
21 일 이내 다음 지원일 비율: 83.7 %
30 일 이내 다음 지원일 비율: 87.1 %


## 6. 최종 대량성 점수

기간별 분포, 연도별 비교, 지원일 간격을 함께 확인해 최종 기준을 7일 14건 이상으로 정한다.

**volume_features**는 유저당 한 행인 최종 대량성 점수표다. 처음에는 user_uuid와 max_7d를 가져와 배점 계산을 확인하고, 전달 파일에는 다른 점수표와 연결하는 데 필요한 user_uuid와 volume_score만 남긴다.

max_7d가 14 이상인지 비교하면 True 또는 False가 나온다. astype(int)는 True를 1점, False를 0점으로 바꾼다. 기준과 정확히 같은 14건도 1점이다.


In [8]:
volume_features = user_period_counts[['max_7d']].reset_index()

volume_features['volume_score'] = (
    volume_features['max_7d'] >= 14
).astype(int)

print(volume_features.head().to_string(index=False))
print('\n대량성 점수별 유저 수')
print(volume_features['volume_score'].value_counts().sort_index().to_string())
print(
    '\n1점 유저 비율:',
    round(volume_features['volume_score'].mean() * 100, 2),
    '%'
)


                           user_uuid  max_7d  volume_score
0002535c-eacb-456b-a620-92c917332ba3       1             0
000b55ad-5c86-438b-a5b0-9ea3a2917789       3             0
0017a738-4fcc-441e-983b-a6033890a9db       1             0
003a83bd-6cb7-4f99-ab3a-66904df762e2       1             0
00495153-f908-4523-9d0d-5b93a57196a4       2             0

대량성 점수별 유저 수
volume_score
0    9742
1     545

1점 유저 비율: 5.3 %


## 7. 다음 분석과 연결

최종 파일에는 유저 ID와 대량성 점수만 남긴다. volume_features.csv를 다른 담당자의 집중성, 다양성, 공고 탐색성, 북마크 탐색성 점수와 user_uuid 기준으로 연결하면 된다.

| 최종 컬럼 | 내용 |
|---|---|
| user_uuid | 다른 점수표와 연결할 유저 ID |
| volume_score | 연속 7일 내 최대 지원 횟수가 14건 이상이면 1, 미만이면 0 |

지원량을 계산할 때는 직무 정보가 없는 지원도 포함한다. 다른 지표와 연결할 때는 팀에서 정한 공통 유저 목록을 기준으로 결합한다. 팀의 최종 유저 수가 이 표와 다르면 공통 분석 대상이 어떻게 정해졌는지 먼저 확인한다.

대량성 1점은 지원량이 큰 행동에 해당한다는 뜻이다. 이후 다른 행동 점수와 합산해 해석한다.


In [9]:
volume_features = volume_features[
    ['user_uuid', 'volume_score']
].copy()

volume_features.to_csv(
    'volume_features.csv',
    index=False,
    encoding='utf-8-sig'
)

print('저장 파일: volume_features.csv')
print('최종 행 수:', len(volume_features))
print('유저 ID 중복 행 수:', volume_features['user_uuid'].duplicated().sum())


저장 파일: volume_features.csv
최종 행 수: 10287
유저 ID 중복 행 수: 0


## 대량성 기준 도출 과정

### 분석 대상과 계산 단위

한국시간 기준 2022년과 2023년의 지원 92,526건, 유저 10,287명을 분석했다. 지원 ID로 구분되는 기록 한 행을 지원 1건으로 계산했다. 같은 유저가 2년 동안 지원한 누적 횟수가 아니라, 각 시점의 연속 기간 내 지원 횟수를 구한 뒤 그 유저의 최대값을 사용했다.

### 횟수 기준 비교

7일, 14일, 21일, 30일에서 각각 유저별 최대 지원 횟수의 분포를 확인했다. 상위 5% 수준을 먼저 살펴보고, Q3 + 1.5×IQR과 Q3 + 3×IQR 기준도 비교했다.

7일 최대 지원 횟수의 95백분위 값은 14건이다. 이를 초과하는 첫 정수인 15건 이상은 487명(4.73%)이다. 반면 14건 이상은 545명(5.30%)이다. 같은 횟수에 여러 유저가 묶이므로, 상위 5% 수준을 횟수 기준으로 바꾸면 정확히 5%가 되지 않는다.

7일의 Q1은 1건, Q3는 4건이므로 IQR은 3건이다. 3×IQR 경계는 4 + 3×3 = 13건이다. 따라서 이 경계를 초과하는 첫 정수인 **14건 이상**을 대량성 후보 기준으로 정했다.

1.5×IQR 기준은 9건 이상으로 1,107명(10.76%)이 해당한다. 일반적인 지원량에서 크게 벗어난 행동을 좁혀 보기 위해 3×IQR 기준을 사용했다. 3이라는 배수는 극단값을 구분하는 분포 규칙으로 선택한 것이며, 허수지원 라벨을 학습해 정한 값은 아니다.

최종 14건은 상위 5%를 정확히 맞추기 위해 정한 수치가 아니라 **Q3 + 3×IQR을 초과하는 횟수**에서 나온 값이다. 그 결과가 약 5% 수준과 가까운지는 보조적으로 확인했다.

| 기간 | Q1 | Q3 | IQR | 3×IQR 기준 | 해당 유저 | 해당 비율 |
|---|---|---|---|---|---|---|
| 7일 | 1 | 4 | 3 | 14건 이상 | 545명 | 5.30% |
| 14일 | 1 | 5 | 4 | 18건 이상 | 452명 | 4.39% |
| 21일 | 1 | 6 | 5 | 22건 이상 | 365명 | 3.55% |
| 30일 | 1 | 6 | 5 | 22건 이상 | 427명 | 4.15% |

### 기간별 연도 비교

기간을 선택하기 위해 2022년과 2023년에서 최대 지원 횟수와 3×IQR 기준을 각각 다시 계산했다.

| 기간 | 2022년 기준 | 2022년 비율 | 2023년 기준 | 2023년 비율 | 비율 차이 |
|---|---|---|---|---|---|
| 7일 | 14건 이상 | 5.17% | 14건 이상 | 5.12% | 0.05%p |
| 14일 | 18건 이상 | 4.46% | 22건 이상 | 2.92% | 1.55%p |
| 21일 | 18건 이상 | 5.18% | 22건 이상 | 3.42% | 1.76%p |
| 30일 | 18건 이상 | 5.64% | 26건 이상 | 3.13% | 2.52%p |

비율 차이는 반올림 전 값으로 계산했다. 표에 표시된 두 비율을 빼면 마지막 자릿수가 다를 수 있다.

7일은 두 해 모두 Q1이 1건, Q3가 4건으로 같았고 횟수 기준도 14건으로 유지됐다. 2022년에는 302명(5.17%), 2023년에는 293명(5.12%)이 해당했다. 비교한 네 기간 중에서 연도별 횟수 기준이 같고 해당 비율도 가장 가까웠다.

14일과 21일은 연도별 기준이 18건과 22건으로 달랐고, 30일은 18건과 26건으로 차이가 더 컸다. 동일한 분포 규칙을 연도별로 다시 적용했을 때의 기준 유지 여부를 기간 선택의 근거로 사용했다.

다만 상위 5% 수준에서 정한 기준을 고정하면 30일도 2022년 4.74%, 2023년 4.61%로 비슷했다. 따라서 해당 비율만으로 7일을 선택한 것은 아니다. 짧은 기간의 대량지원 행동을 보려는 목적과 3×IQR 기준의 연도별 일관성을 함께 고려했다.

연도별 비율의 분모는 해당 연도의 지원 유저 수다. 두 해에 모두 지원한 유저가 있으므로 302명과 293명을 더해 전체 해당 유저 수로 해석하지 않는다.

### 지원일 간격 확인

같은 유저가 지원한 서로 다른 날짜 사이의 간격을 계산한 결과 중앙값은 6일이었다. 다음 지원일이 7일 이내인 간격은 63.1%, 14일 이내는 78.5%, 30일 이내는 87.1%였다.

중앙값 6일은 주 단위로 지원량을 관찰하는 데 참고할 수 있는 결과다. 간격만으로 최적 기간을 결정하지 않고, 기간별 분포와 연도별 비교 결과를 함께 보고 7일을 선택했다. 이 비율은 유저 수의 비율이 아닌 지원일 사이 간격의 비율이다.

### 최종 배점과 전달

**유저별 연속 7일 내 최대 지원 횟수 max_7d가 14건 이상이면 volume_score에 1점, 미만이면 0점을 부여한다.**

이 기준으로 1점을 받은 유저는 545명(5.30%)이다. 최종 volume_features.csv에는 user_uuid와 volume_score를 남겼다. 다른 담당자의 점수표와 user_uuid 기준으로 연결하고, 대량성은 전체 행동 점수 중 1점을 담당한다.

기간과 횟수 기준은 지원량이 큰 행동을 구분하기 위한 선택이다. 실제 허수지원 여부는 다른 행동 지표와 함께 해석한다.
